# Task 6 — Idempotent Replay Verification

## Mục tiêu

Reprocess một file không đổi, sửa file đó rồi reprocess, restart Spark và cuối
cùng phục hồi source/database về revision gốc mà không tạo dữ liệu trùng hoặc
stale.

## Thiết kế và triển khai

[`task6/replay_single_file.py`](https://github.com/linh285/huggingface-cpg-streaming/blob/main/task6/replay_single_file.py) so sánh
exact node/edge ID sets cho từng pha, kiểm count toàn graph, giữ
`file_id` ổn định và dùng `content_sha256` làm revision. Trước restart, script
chụp `(_id, content_sha256, kafka_offset, processed_at)` của đủ 147 document;
sau khi query Spark trở lại `RUNNING`, script poll collection liên tục trong
15 giây và toàn bộ snapshot phải giữ nguyên.

## Lệnh thực thi

```bash
python task6/replay_single_file.py \
  --file src/datasets/utils/experimental.py --timeout 600
```

## Kết quả thực nghiệm


In [1]:
import json
from pathlib import Path

report = json.loads(Path("../artifacts/task6/replay_result.json").read_text(encoding="utf-8"))
phases = report["phase_comparison"]
assert report["status"] == "PASS"
assert report["file_id_stable_across_revisions"] is True
assert report["duplicate_node_ids"] == 0
assert report["duplicate_edge_ids"] == 0
assert phases["baseline"] == {
    "file_nodes": 57, "file_edges": 75,
    "global_nodes": 208003, "global_edges": 267695,
    "mongo_documents": 147,
}
assert phases["unchanged_replay"] == phases["baseline"]
assert phases["modified_replay"] == {
    "file_nodes": 68, "file_edges": 89,
    "global_nodes": 208014, "global_edges": 267709,
    "mongo_documents": 147,
}
assert phases["restart"] == {
    "mongo_documents_before": 147,
    "mongo_documents_after": 147,
    "snapshots_equal": True,
}
assert phases["cleanup"] == phases["baseline"]

print("phase              file N/E   global N/E       Mongo docs")
for name in ("baseline", "unchanged_replay", "modified_replay", "cleanup"):
    phase = phases[name]
    print(
        f"{name:<18} {phase['file_nodes']:>3}/{phase['file_edges']:<3} "
        f"{phase['global_nodes']:>6}/{phase['global_edges']:<6} "
        f"{phase['mongo_documents']:>10}"
    )
print("\nrestart:", json.dumps(phases["restart"], indent=2))
print(
    "checkpoint files:",
    report["spark_restart"]["checkpoint_files_before"],
    "->",
    report["spark_restart"]["checkpoint_files_after"],
)
print(
    "restart observation:",
    report["spark_restart"]["observation_seconds"],
    "seconds /",
    report["spark_restart"]["observation_checks"],
    "snapshot checks",
)
print("original hash:", report["original_content_sha256"])
print("modified hash:", report["modified_content_sha256"])
print("cleanup hash:", report["cleanup_restore"]["mongo_content_sha256"])


phase              file N/E   global N/E       Mongo docs
baseline            57/75  208003/267695        147
unchanged_replay    57/75  208003/267695        147
modified_replay     68/89  208014/267709        147
cleanup             57/75  208003/267695        147

restart: {
  "mongo_documents_before": 147,
  "mongo_documents_after": 147,
  "snapshots_equal": true
}
checkpoint files: 104 -> 104
restart observation: 15 seconds / 9 snapshot checks
original hash: 6acb052a2ded7705891b4be5a06fba143eb2881e75612fa606bc31fe8ef677b3
modified hash: e360b90ebfe2e1281e55872ae7199d71c72c777a580f5d05c626f565ccc32482
cleanup hash: 6acb052a2ded7705891b4be5a06fba143eb2881e75612fa606bc31fe8ef677b3


![Neo4j sau cleanup](images/task4/neo4j-full-corpus-counts.jpg)

![Spark query sau restart](images/task5/spark-full-corpus-running.jpg)

![MongoDB sau cleanup](images/task5/mongodb-full-corpus.jpg)

## Vấn đề và cách xử lý

Chỉ kiểm count từng file có thể bỏ sót tác động lên phần còn lại của corpus.
Script được mở rộng để kiểm cả per-file và global count, bắt duplicate trên
toàn graph và so sánh 147 document qua restart. Khối `finally` phục hồi file và
publish revision gốc ngay cả khi một pha trước lỗi.

## Đánh giá

Baseline và cleanup đều là 57/75 cho file, 208.003/267.695 toàn graph; revision
tạm là 68/89 và 208.014/267.709. MongoDB giữ 147 document trong mọi pha, hai
snapshot restart giống nhau và hash cleanup trở về hash gốc. Test hiện phụ thuộc
stack local còn đủ RAM và các connector ở trạng thái `RUNNING`.
